
# Classroom Workshop

This workshop uses a synthetic e-commerce dataset modelled after a Southeast Asian online retail platform. It is designed to teach star schema analytics — a data warehouse pattern where a central fact table (transactional events) is surrounded by dimension tables (descriptive context). All queries execute in Databricks SQL against Delta Lake tables in Unity Catalog.

## Data Ingestion

Follow these steps in Databricks Free Edition before running any SQL cells.

###Create Schema & Tables in Databricks

Step 1 — Create the Schema (ecommerce)

- Navigate: Left sidebar → Catalog icon → Click workspace catalog.
- Click the Create Schema button (or ➕ icon next to the workspace catalog).
- Enter schema name: ecommerce. Leave Storage Location as default. Click Create.



Step 2 — Create the Volume (raw)
- Navigate: Catalog → workspace → ecommerce → Volumes tab → Create Volume.
- Enter volume name: raw. Volume type: Managed. Click Create.
- Your volume is now at: /Volumes/workspace/ecommerce/raw 

Step 3 — Verify via SQL

In [0]:
%sql
USE workspace.ecommerce;

In [0]:
# ecommerce should appear
SHOW SCHEMAS IN workspace;   

In [0]:
# Volume raw should appear
SHOW VOLUMES IN workspace.ecommerce;  

Step 4 — Create Delta Tables

Open a SQL notebook and run the DDL below. 

PRIMARY KEY and FOREIGN KEY declarations are supported by Databricks but NOT ENFORCED — they document the schema and aid the query optimizer.

DIMENSION: customers

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.ecommerce.customers (
  customer_id       INT     NOT NULL,
  first_name        STRING,
  last_name         STRING,
  email             STRING,
  city              STRING,
  country           STRING,
  registration_date DATE,
  age               INT,
  gender            STRING,
  membership_tier   STRING,
  PRIMARY KEY (customer_id) NOT ENFORCED
) USING DELTA;


DIMENSION: products 

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.ecommerce.products (
  product_id   INT    NOT NULL,
  product_name STRING,
  category     STRING,
  brand        STRING,
  unit_price   DOUBLE,
  unit_cost    DOUBLE,
  PRIMARY KEY (product_id) NOT ENFORCED
) USING DELTA;


BRIDGE: orders (connects customers to order_items)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.ecommerce.orders (
  order_id         INT    NOT NULL,
  customer_id      INT,        -- FK to customers.customer_id
  order_date       DATE,
  status           STRING,
  payment_method   STRING,
  shipping_city    STRING,
  shipping_country STRING,
  PRIMARY KEY (order_id) NOT ENFORCED,
  FOREIGN KEY (customer_id)
    REFERENCES workspace.ecommerce.customers(customer_id) NOT ENFORCED
) USING DELTA;


FACT: order_items 

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.ecommerce.order_items (
  order_item_id INT    NOT NULL,
  order_id      INT,        -- FK to orders.order_id
  product_id    INT,        -- FK to products.product_id
  quantity      INT,
  unit_price    DOUBLE,
  discount_pct  INT,
  PRIMARY KEY (order_item_id) NOT ENFORCED,
  FOREIGN KEY (order_id)
    REFERENCES workspace.ecommerce.orders(order_id)     NOT ENFORCED,
  FOREIGN KEY (product_id)
    REFERENCES workspace.ecommerce.products(product_id) NOT ENFORCED
) USING DELTA;


### Load CSV Data into Delta Tables

Upload all four CSVs to /Volumes/workspace/ecommerce/raw via the Catalog UI, then run COPY INTO in dependency order.


Step 1: load dimensions first (no FK dependencies)

In [0]:
COPY INTO workspace.ecommerce.customers
FROM '/Volumes/workspace/ecommerce/raw/customers.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true');

COPY INTO workspace.ecommerce.products
FROM '/Volumes/workspace/ecommerce/raw/products.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true');


Step 2: bridge table (depends on customers)

In [0]:
COPY INTO workspace.ecommerce.orders
FROM '/Volumes/workspace/ecommerce/raw/orders.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true');


Step 3: fact table last (depends on orders and products)


In [0]:
COPY INTO workspace.ecommerce.order_items
FROM '/Volumes/workspace/ecommerce/raw/order_items.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true');


### Verify

Connection check: Run this quick JOIN validation to confirm orders and customers connect correctly: SELECT COUNT(*) FROM orders o JOIN customers c ON o.customer_id = c.customer_id;  Expected result: 200 rows — matching all 200 orders to a customer.

In [0]:
%sql
SELECT 'customers'   AS tbl, COUNT(*) AS rows FROM customers
UNION ALL
SELECT 'products',          COUNT(*)         FROM products
UNION ALL
SELECT 'orders',            COUNT(*)         FROM orders
UNION ALL
SELECT 'order_items',       COUNT(*)         FROM order_items
ORDER BY rows DESC;


## Basic Aggregations

These queries operate on single tables or a simple two-table join. They introduce COUNT, SUM, AVG, GROUP BY, and HAVING — the five building blocks of every analytical query.

### Total Revenue and Order Volume

A scalar aggregate (no GROUP BY) gives a platform-wide health check. The net revenue formula accounts for both quantity and discount_pct from the fact table.

From the order_items table, calculate the following sales performance metrics from order data:
- Total Line Items: Count every individual row (item) listed in the table.
- Total Orders: Count how many unique order IDs exist (ignoring duplicates where one order has multiple items).
- Units Sold: Add up the total quantity of all items sold.
- Gross Revenue: Calculate the total sales value by multiplying quantity by unit price for every row, then round the result to two decimal places.
- Net Revenue: Calculate the total sales value after applying discounts. It takes the gross value and subtracts the percentage defined in discount_pct, then rounds the result to two decimal places.
- Average Unit Price: Find the average price per item across all rows and round it to two decimal places.


In [0]:
%sql


### Sales Performance By Category
For every unique product category, calculate the following:
- Line Items: The total number of individual entries or "rows" sold in that category.
- Units Sold: The total physical quantity of products sold.
- Gross Revenue: The total sales amount (Quantity × Price) before any discounts, rounded to two decimal places.
- Net Revenue: The total sales amount after applying the discount percentage, rounded to two decimal places.
- Average Price: The average price of an item within that category.
- Group By: Bundle all the math above so you get one row of results for each product category.
- Order By: Sort the entire list by Net Revenue, putting the highest-earning categories at the top.

In [0]:
%sql


### Orders By Payment Method

For all the orders, group every transaction based on the payment_method used (e.g., Credit Card, PayPal, Apple Pay). 

For each payment method, it calculates:
- Order Count: The total number of unique orders placed using that specific method.
- Percentage of Orders: The "market share" of that payment method. It takes the count for that specific method, divides it by the grand total of all orders across the entire table, and multiplies by 100 to get a percentage. This is rounded to one decimal place (e.g., 25.5%).
- Sort the list by the Order Count in descending order.

In [0]:
%sql


Great! You have completed the workshop on time. Down this notebook and submit in the canvas assignment. 

Take a break. 